# Google GenAI SDK 使用指南

本 Notebook 演示如何通过 **七牛 AIToken** 平台，使用 **Google GenAI SDK** (`google-genai`) 调用 Gemini 系列模型。

## 功能简介

Google GenAI SDK 是 Google 官方提供的 AI 模型调用工具包，支持通过 Vertex AI 协议访问 Gemini 模型。

- **流式生成（Streaming）**：实时逐步接收模型输出
- **非流式生成（Non-streaming）**：一次性获取完整回复
- **多轮对话（Multi-turn Chat）**：维护对话上下文，进行连续对话
- **系统指令（System Instruction）**：通过系统指令定义模型角色和行为

## 1. 环境准备

安装 `google-genai` 依赖包，并配置客户端指向七牛 AIToken 平台。

> **前置条件**：需要 Python 3.9 或更高版本，并设置环境变量 `QINIU_API_KEY`。

In [ ]:
# 安装 Google GenAI SDK
%pip install google-genai -q

In [ ]:
import os
from google import genai
from google.genai import types
from google.genai.types import HttpOptions

# ==================== 配置参数 ====================
MODEL_ID = "gemini-3-pro-preview"  # 模型 ID
BASE_URL = "https://api.qnaigc.com/bypass/vertex"  # 七牛 AIToken 国内节点
# ==================================================

# 从环境变量读取 API Key
api_key = os.environ.get("QINIU_API_KEY", "<your-api-key>")

# 初始化客户端
# 使用 vertexai=True 并通过 HttpOptions 配置自定义端点和认证
client = genai.Client(
    vertexai=True,
    http_options=HttpOptions(
        api_version="v1",
        base_url=BASE_URL,
        headers={"Authorization": f"Bearer {api_key}"},
    ),
)

print("环境配置完成!")
print(f"  API 端点: {BASE_URL}")
print(f"  模型: {MODEL_ID}")

## 2. 示例一：流式文本生成（Streaming）

使用 `generate_content_stream()` 方法进行流式文本生成，模型输出会逐步返回，适合实时展示。

In [ ]:
# 流式文本生成
prompt = "请用三句话介绍量子计算的基本原理。"
print(f"Prompt: {prompt}\n")
print("-" * 50)

response = client.models.generate_content_stream(
    model=MODEL_ID,
    contents=prompt,
    config=types.GenerateContentConfig(
        temperature=0.7,
    ),
)

print("Response (Streaming):")
for chunk in response:
    if chunk.text:
        print(chunk.text, end="", flush=True)
print("\n")
print("-" * 50)

## 3. 示例二：非流式文本生成（Non-streaming）

使用 `generate_content()` 方法一次性获取完整回复，适合不需要实时展示的场景。

In [ ]:
# 非流式文本生成
prompt = "列举 Python 中最常用的 5 个内置数据结构，并各用一句话说明。"
print(f"Prompt: {prompt}\n")
print("-" * 50)

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config=types.GenerateContentConfig(
        temperature=0.3,
    ),
)

print("Response:")
print(response.text)
print("-" * 50)

# 打印 token 用量信息
if response.usage_metadata:
    print(f"\nToken 用量:")
    print(f"  输入 tokens: {response.usage_metadata.prompt_token_count}")
    print(f"  输出 tokens: {response.usage_metadata.candidates_token_count}")
    print(f"  总计 tokens: {response.usage_metadata.total_token_count}")

## 4. 示例三：自定义生成参数

通过 `GenerateContentConfig` 可以精细控制生成行为，包括温度、Top-P、最大输出长度、停止序列等。

In [ ]:
# 自定义生成参数
prompt = "写一首关于春天的五言绝句。"
print(f"Prompt: {prompt}\n")
print("-" * 50)

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config=types.GenerateContentConfig(
        temperature=0.9,        # 较高温度，增加创意性
        top_p=0.95,             # Top-P 采样
        max_output_tokens=256,  # 最大输出 token 数
    ),
)

print("Response:")
print(response.text)
print("-" * 50)

## 5. 示例四：系统指令（System Instruction）

通过 `system_instruction` 参数可以为模型设定角色和行为规范，影响后续所有回复的风格。

In [ ]:
# 使用系统指令定义模型角色
prompt = "Redis 和 Memcached 有什么区别？"
print(f"Prompt: {prompt}\n")
print("-" * 50)

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config=types.GenerateContentConfig(
        temperature=0.3,
        system_instruction="你是一位资深后端架构师，回答问题时请结合实际项目经验，语言简洁专业。",
    ),
)

print("Response:")
print(response.text)
print("-" * 50)

## 6. 示例五：多轮对话（Multi-turn Chat）

通过构造包含多条消息的 `contents` 列表，可以实现多轮对话。模型会根据上下文进行连贯回复。

In [ ]:
# 多轮对话
# 构造对话历史
contents = [
    types.Content(role="user", parts=[types.Part.from_text(text="你好，我想学习 Go 语言，应该从哪里开始？")]),
    types.Content(role="model", parts=[types.Part.from_text(text="你好！学习 Go 语言建议从以下几步开始：\n1. 安装 Go 环境并配置 GOPATH\n2. 阅读官方 Tour of Go 教程\n3. 学习基础语法：变量、函数、结构体、接口\n4. 实践一个小项目，比如写一个简单的 HTTP 服务")]),
    types.Content(role="user", parts=[types.Part.from_text(text="能详细说说第 4 步吗？给一个具体的例子。")]),
]

print("多轮对话 - 第三轮回复:\n")
print("-" * 50)

response = client.models.generate_content(
    model=MODEL_ID,
    contents=contents,
    config=types.GenerateContentConfig(
        temperature=0.5,
    ),
)

print("Response:")
print(response.text)
print("-" * 50)

## 7. 参数说明

### 客户端配置

| 参数 | 说明 |
|------|------|
| `vertexai` | 设为 `True`，使用 Vertex AI 协议 |
| `http_options.api_version` | API 版本，固定为 `"v1"` |
| `http_options.base_url` | API 端点地址，设为 `https://api.qnaigc.com/bypass/vertex` |
| `http_options.headers` | 请求头，需包含 `Authorization: Bearer <API_KEY>` |

### GenerateContentConfig 主要参数

| 参数 | 类型 | 说明 |
|------|------|------|
| `temperature` | float | 生成温度，范围 0.0-2.0，越高越随机 |
| `top_p` | float | Top-P 采样，范围 0.0-1.0 |
| `top_k` | int | Top-K 采样 |
| `max_output_tokens` | int | 最大输出 token 数 |
| `stop_sequences` | list[str] | 停止序列，遇到时停止生成 |
| `system_instruction` | str | 系统指令，定义模型角色和行为 |

### 环境变量

| 环境变量 | 说明 |
|----------|------|
| `QINIU_API_KEY` | 七牛 AIToken 平台的 API Key |

## 8. 注意事项

1. **协议模式**：本示例使用 `vertexai=True` 模式，通过七牛 AIToken 平台代理访问 Gemini 模型
2. **认证方式**：通过 `HttpOptions.headers` 传递 Bearer Token 认证，而非 Google Cloud 默认的 ADC 认证
3. **模型选择**：七牛 AIToken 平台支持的 Gemini 模型请参考平台文档，当前示例使用 `gemini-3-pro-preview`
4. **流式 vs 非流式**：流式适合实时交互场景（如聊天），非流式适合批处理场景

## 更多资源

- [七牛 AIToken 文档](https://developer.qiniu.com/aitokenapi)
- [Google GenAI SDK 文档](https://googleapis.github.io/python-genai/)
- [Gemini API 文档](https://ai.google.dev/gemini-api/docs)
- [七牛 API 调用示例文档](https://apidocs.qnaigc.com)